# 002 multilayer perceptron (mlp)

The Iris dataset contains 150 samples of flowers, with 4 input features (petal/sepal dimensions) and 3 output classes (species). Neural networks work best with normalized data and one-hot encoded labels.

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# 1. Load the dataset
iris = load_iris()
X = iris.data
y = iris.target.reshape(-1, 1)

# 2. One-hot encode the target labels (e.g., class 2 becomes [0, 0, 1])
encoder = OneHotEncoder(sparse_output=False)
y_hot = encoder.fit_transform(y)

# 3. Standardize the inputs (mean=0, variance=1) for faster, stable convergence
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Split into 80% training and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_hot, test_size=0.2, random_state=42)

print(f"Training features shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")

Training features shape: (120, 4)
Training labels shape: (120, 3)


# Defining the Multilayer Perceptron
Here, we will build a 3-layer architecture: an input layer (4 neurons), one hidden layer (we'll use 8 neurons), and an output layer (3 neurons).

We will use a standard Sigmoid activation function. To update the weights, we use gradient descent via backpropagation, which ultimately boils down to computing the error and multiplying it by the derivative of our activation function.

In [ ]:
class MultilayerPerceptron:
    def __init__(self, input_size, hidden_size, output_size):
        # Initialize weights randomly and biases to zero
        # We scale weights by 0.1 to keep initial outputs in a reasonable range
        self.W1 = np.random.randn(input_size, hidden_size) * 0.1
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.1
        self.b2 = np.zeros((1, output_size))

    def sigmoid(self, z):
        """Activation function mapping values to a curve between 0 and 1."""
        return 1 / (1 + np.exp(-z))

    def sigmoid_derivative(self, a):
        """Derivative of the sigmoid function, optimized to take the post-activation output."""
        return a * (1 - a)

    def forward(self, X):
        """Passes data forward through the network using dot products."""
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = self.sigmoid(self.z1)
        
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = self.sigmoid(self.z2)
        return self.a2

    def backward(self, X, y, output, learning_rate):
        """Propagates the error backwards to update weights and biases."""
        # 1. Calculate the error at the output layer
        output_error = y - output
        output_delta = output_error * self.sigmoid_derivative(output)

        # 2. Calculate the error at the hidden layer 
        # (How much did the hidden layer contribute to the output error?)
        hidden_error = np.dot(output_delta, self.W2.T)
        hidden_delta = hidden_error * self.sigmoid_derivative(self.a1)

        # 3. Update weights and biases (Gradient Ascent on the error)
        self.W2 += np.dot(self.a1.T, output_delta) * learning_rate
        self.b2 += np.sum(output_delta, axis=0, keepdims=True) * learning_rate
        
        self.W1 += np.dot(X.T, hidden_delta) * learning_rate
        self.b1 += np.sum(hidden_delta, axis=0, keepdims=True) * learning_rate

    def train(self, X, y, epochs, learning_rate=0.05):
        for i in range(epochs):
            output = self.forward(X)
            self.backward(X, y, output, learning_rate)
            
            if i % 1000 == 0:
                loss = np.mean(np.square(y - output)) # Mean Squared Error
                print(f"Epoch {i:4d} | MSE Loss: {loss:.4f}")

# Training and Evaluation
Now we instantiate the model, train it on our data, and write a quick function to test its accuracy by comparing the network's highest predicted probability against the actual label.

In [ ]:
# 4 inputs, 8 hidden neurons, 3 outputs (classes)
mlp = MultilayerPerceptron(input_size=4, hidden_size=8, output_size=3)

print("Starting training...")
mlp.train(X_train, y_train, epochs=5000, learning_rate=0.01)
print("Training complete.\n")

# Make predictions on the test set
predictions = mlp.forward(X_test)

# Convert probabilities back to class labels (0, 1, or 2)
predicted_classes = np.argmax(predictions, axis=1)
actual_classes = np.argmax(y_test, axis=1)

# Calculate accuracy
accuracy = np.mean(predicted_classes == actual_classes) * 100
print(f"Test Set Accuracy: {accuracy:.2f}%")

# Display a few comparisons
print("\nSample predictions:")
for i in range(5):
    print(f"Predicted: {predicted_classes[i]}, Actual: {actual_classes[i]}")

# Next Steps
swap in a ReLU activation function